In [1]:
pip install scikit-learn xgboost joblib

Step 1 — Load the feature-engineered dataset

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [9]:
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',                        # semicolon-separated, not comma
    na_values='?',                  # '?' marks missing values in this file
    parse_dates={'Datetime': ['Date', 'Time']},  # merge Date + Time into one column
    dayfirst=True                   # date format is dd/mm/yyyy
)

df = df.set_index('Datetime').sort_index()
df = df.dropna()                    # drop the ~1.25% missing rows

/tmp/ipykernel_2313/738480049.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(


In [17]:
# Resample from minute-level to hourly averages
# Brings 2 million rows down to ~35,000 — much faster to train on
df = df.resample('h').mean()
df = df.dropna() # Drop any NaNs introduced by resampling

# Engineer time features
df['Hour']       = df.index.hour
df['Day']        = df.index.day
df['Month']      = df.index.month
df['Weekday']    = df.index.dayofweek
df['Is_weekend'] = (df.index.dayofweek >= 5).astype(int)

In [11]:
TARGET   = 'Global_active_power'

FEATURES = [
    'Voltage', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
    'Hour', 'Day', 'Month', 'Weekday', 'Is_weekend'
]

X = df[FEATURES]
y = df[TARGET]

Step 2 — Define features (X) and target (y)

In [22]:
# Target
TARGET = 'Global_active_power'

In [23]:
#Input features
FEATURES = [
'Voltage', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
    'Hour', 'Day', 'Month', 'Weekday', 'Is_weekend'
]

X = df[FEATURES]
y = df[TARGET]

Step 3 — Chronological split (the critical step)


In [24]:
# Compute the split index — 80% of rows by position (not random)
split_idx = int(len(df) * 0.80)

# Split maintaining time order
X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

# Verify the boundary date
print(f'Training: {X_train.index.min()} to {X_train.index.max()}')
print(f'Testing:  {X_test.index.min()}  to {X_test.index.max()}')
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')


Training: 2006-12-16 17:00:00 to 2010-02-05 04:00:00
Testing:  2010-02-05 05:00:00  to 2010-11-26 21:00:00
Train rows: 27334 | Test rows: 6834


In [26]:
#drop missing value
train = X_train.copy()
train['target'] = y_train

train = train.dropna()

X_train = train.drop('target', axis=1)
y_train = train['target']

Step 4 — Train all three models

In [27]:
# Model 1: Linear Regression (baseline)
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                           min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Model 3: XGBoost
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8, random_state=42,
                   early_stopping_rounds=20, eval_metric='rmse')

xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = xgb.predict(X_test)


Step 5 — Compute all evaluation metrics

In [28]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name:25s}  MAE={mae:.4f} kW  |  RMSE={rmse:.4f} kW  |  R²={r2:.4f}')

evaluate('Linear Regression',   y_test, lr_pred)
evaluate('Random Forest',        y_test, rf_pred)
evaluate('XGBoost',              y_test, xgb_pred)


Linear Regression          MAE=0.2631 kW  |  RMSE=0.3368 kW  |  R²=0.7949
Random Forest              MAE=0.1775 kW  |  RMSE=0.2696 kW  |  R²=0.8686
XGBoost                    MAE=0.1721 kW  |  RMSE=0.2588 kW  |  R²=0.8790


Step 6 — Save the best model for the dashboard

In [30]:
import joblib
import os

# Create the 'dashboard' directory if it doesn't exist
os.makedirs('dashboard', exist_ok=True)

# Save Random Forest (or XGBoost — whichever scores best)
joblib.dump(rf, 'dashboard/rf_model.pkl')
joblib.dump(xgb, 'dashboard/xgb_model.pkl')
print('Models saved.')

Models saved.
